In [6]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # Masque les warnings généraux
warnings.filterwarnings("ignore", category=FutureWarning) # Masque les FutureWarnings
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)  # Masque ton SettingWithCopyWarning spécifique


In [7]:
# Cellule 1 : Imports de base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Environnement prêt !")
print("📊 Pandas version:", pd.__version__)

✅ Environnement prêt !
📊 Pandas version: 2.2.3


In [8]:
df = pd.read_csv(r'C:\Users\Achraf Alaabouch\Desktop\mon_projet_immobilier\data\raw\mubawab_listings.csv')

print("📈 Shape du dataset:", df.shape)
print("\n👀 Premières lignes:")
df.head()


📈 Shape du dataset: (3300, 7)

👀 Premières lignes:


,Type,Localisation,Latitude,Longitude,Title,Price,Tags
0,Appartements Casablanca,Casablanca Finance City à Casablanca,NaN,NaN,Somptueux appartement à louer au tour végétal,22 000 DH,"['219 m²', '4 Pièces', '3 Chambres', '3 Salles..."
1,Appartements Casablanca,Anfa à Casablanca,NaN,NaN,A vendre appartement 86m face ELBILIA lahjajma,1 230 000 DH,"['86 m²', '3 Pièces', '2 Chambres', '2 Salles ..."
2,Appartements Casablanca,La Gironde à Casablanca,NaN,NaN,A vendre bel appartement 2 chambres Gironde,900 000 DH,"['86 m²', '3 Pièces', '2 Chambres', '1 Salle d..."
3,Appartements Casablanca,Bourgogne Ouest à Casablanca,NaN,NaN,A vendre appartement 180m clinique Badr pas cher,2 400 000 DH,"['180 m²', '5 Pièces', '3 Chambres', '3 Salles..."
4,Appartements Casablanca,Les princesses à Casablanca,33.574209,-7.644182,Appartement 3 chambres yaacoub almansour pas cher,1 550 000 DH,"['120 m²', '5 Pièces', '3 Chambres', '2 Salles..."


In [9]:
# NETTOYAGE COMPLET
print("🧹 DEBUT DU NETTOYAGE...")
print("Initial:", df.shape)

# 1. Supprimer lignes sans prix
df_clean = df.dropna(subset=['Price'])
print("Après suppression prix manquants:", df_clean.shape)

# 2. Remplacer types manquants
type_frequent = df_clean['Type'].mode()[0]
df_clean['Type'] = df_clean['Type'].fillna(type_frequent)
print(f"Types manquants → '{type_frequent}'")

# 3. Vérification finale
print("\n✅ NETTOYAGE TERMINE!")
print("Valeurs manquantes restantes:")
print(df_clean.isnull().sum())

🧹 DEBUT DU NETTOYAGE...
Initial: (3300, 7)
Après suppression prix manquants: (3134, 7)
Types manquants → 'Appartements Casablanca'

✅ NETTOYAGE TERMINE!
Valeurs manquantes restantes:
Type               0
Localisation       0
Latitude        1445
Longitude       1445
Title              0
Price              0
Tags               0
dtype: int64


In [10]:
# Cellule 6 : Voir un exemple de Tags
exemple_tags = df_clean['Tags'].iloc[0]
print("🔍 Exemple de Tags:")
print(exemple_tags)
print("\n📝 Type:", type(exemple_tags))

🔍 Exemple de Tags:
['219 m²', '4 Pièces', '3 Chambres', '3 Salles de bains', 'Nouveau', "Moins d'un an", '11ème étage']

📝 Type: <class 'str'>


In [11]:
# Cellule 7 : Fonction d'extraction des Tags
import ast  # Pour convertir la string en liste

def extraire_tags(tags_string):
    """
    Transforme : "['219 m²', '4 Pièces', '3 Chambres']"
    En dictionnaire : {'surface': 219, 'pieces': 4, 'chambres': 3}
    """
    try:
        # Convertir la string en liste Python
        tags_list = ast.literal_eval(tags_string)
        
        result = {}
        
        for tag in tags_list:
            # Surface
            if 'm²' in tag:
                result['surface'] = int(tag.split(' ')[0])
            # Pièces
            elif 'Pièces' in tag or 'Pièce' in tag:
                result['pieces'] = int(tag.split(' ')[0])
            # Chambres
            elif 'Chambres' in tag or 'Chambre' in tag:
                result['chambres'] = int(tag.split(' ')[0])
            # Salles de bain
            elif 'Salles de bains' in tag or 'Salle de bain' in tag:
                result['salles_bain'] = int(tag.split(' ')[0])
            # Étage
            elif 'étage' in tag:
                etage_str = tag.split(' ')[0]
                if 'ème' in etage_str:
                    result['etage'] = int(etage_str.replace('ème', ''))
                elif 'er' in etage_str:
                    result['etage'] = 1
            # État
            elif tag in ['Nouveau', 'Bon état', 'Très bon état', 'À rénover']:
                result['etat'] = tag
            # Âge
            elif 'ans' in tag or 'an' in tag:
                result['age'] = tag
        
        return result
        
    except:
        return {}

In [12]:
# Cellule 8 : Tester sur le premier exemple
tags_test = df_clean['Tags'].iloc[0]
resultat_test = extraire_tags(tags_test)

print("🧪 Test de la fonction:")
print("Tags originaux:", tags_test)
print("Tags extraits:", resultat_test)

🧪 Test de la fonction:
Tags originaux: ['219 m²', '4 Pièces', '3 Chambres', '3 Salles de bains', 'Nouveau', "Moins d'un an", '11ème étage']
Tags extraits: {'surface': 219, 'pieces': 4, 'chambres': 3, 'salles_bain': 3, 'etat': 'Nouveau', 'age': "Moins d'un an", 'etage': 11}


In [13]:
# Cellule 9 : Appliquer à toutes les lignes
print("🔄 Extraction des tags pour toutes les lignes...")

# Appliquer la fonction à toute la colonne Tags
tags_extraits = df_clean['Tags'].apply(extraire_tags)

# Créer de nouvelles colonnes
df_clean['surface'] = tags_extraits.apply(lambda x: x.get('surface'))
df_clean['pieces'] = tags_extraits.apply(lambda x: x.get('pieces'))
df_clean['chambres'] = tags_extraits.apply(lambda x: x.get('chambres'))
df_clean['salles_bain'] = tags_extraits.apply(lambda x: x.get('salles_bain'))
df_clean['etage'] = tags_extraits.apply(lambda x: x.get('etage'))
df_clean['etat'] = tags_extraits.apply(lambda x: x.get('etat'))
df_clean['age'] = tags_extraits.apply(lambda x: x.get('age'))

print("✅ Nouvelles colonnes créées!")

🔄 Extraction des tags pour toutes les lignes...
✅ Nouvelles colonnes créées!


In [14]:
# Cellule 10 : Vérification
print("📊 Dataset avec nouvelles colonnes:")
print(df_clean[['Price', 'surface', 'pieces', 'chambres', 'salles_bain', 'etage']].head())

print(f"\n📈 Statistiques extraction:")
print(f"Surfaces extraites: {df_clean['surface'].notna().sum()}/{len(df_clean)}")
print(f"Chambres extraites: {df_clean['chambres'].notna().sum()}/{len(df_clean)}")

📊 Dataset avec nouvelles colonnes:
          Price  surface  pieces  chambres  salles_bain  etage
0     22 000 DH      219     4.0       3.0          3.0   11.0
1  1 230 000 DH       86     3.0       2.0          2.0    1.0
2    900 000 DH       86     3.0       2.0          1.0    5.0
3  2 400 000 DH      180     5.0       3.0          3.0    3.0
4  1 550 000 DH      120     5.0       3.0          2.0    5.0

📈 Statistiques extraction:
Surfaces extraites: 3134/3134
Chambres extraites: 3125/3134


In [15]:
# Cellule 11 : Nettoyer les prix (Gère DH et EUR)
def nettoyer_prix(prix_string):
    """
    Transforme "22 000 DH" en 22000
    Transforme "2 000 EUR" en 2000 * 11 (conversion EUR → DH)
    """
    if isinstance(prix_string, str):
        # Vérifier si c'est en EUR ou DH
        if 'EUR' in prix_string:
            # Convertir EUR en DH (1 EUR ≈ 11 DH)
            valeur = int(prix_string.replace('EUR', '').replace(' ', '').strip())
            return valeur * 11
        elif 'DH' in prix_string:
            # Prix déjà en DH
            return int(prix_string.replace('DH', '').replace(' ', '').strip())
        else:
            # Si pas d'unité, supposer que c'est en DH
            return int(prix_string.replace(' ', '').strip())
    return prix_string

# Test sur quelques valeurs pour voir les différents formats
print("🧪 Test de nettoyage prix:")
test_prix = df_clean['Price'].head(10)
for prix in test_prix:
    print(f"'{prix}' → {nettoyer_prix(prix)}")

# Appliquer à tout le dataset
df_clean['prix_numerique'] = df_clean['Price'].apply(nettoyer_prix)

print("\n💰 Prix convertis en numérique:")
print(df_clean[['Price', 'prix_numerique']].head())

🧪 Test de nettoyage prix:
'22 000 DH' → 22000
'1 230 000 DH' → 1230000
'900 000 DH' → 900000
'2 400 000 DH' → 2400000
'1 550 000 DH' → 1550000
'1 450 000 DH' → 1450000
'4 800 000 DH' → 4800000
'1 700 000 DH' → 1700000
'330 000 DH' → 330000
'7 500 000 DH' → 7500000

💰 Prix convertis en numérique:
          Price  prix_numerique
0     22 000 DH           22000
1  1 230 000 DH         1230000
2    900 000 DH          900000
3  2 400 000 DH         2400000
4  1 550 000 DH         1550000


In [16]:
# Cellule 12 : Préparation pour Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Supprimer les lignes avec surface manquante (essentielle pour le modèle)
df_model = df_clean.dropna(subset=['surface', 'prix_numerique'])

# 2. Sélectionner les features importantes
features = ['surface', 'pieces', 'chambres', 'salles_bain', 'etage']
X = df_model[features]
y = df_model['prix_numerique']

# 3. Remplacer les valeurs manquantes
X = X.fillna(X.median())

print(f"📊 Données pour l'entraînement: {X.shape}")
print(f"🎯 Variable cible: {y.shape}")

📊 Données pour l'entraînement: (3134, 5)
🎯 Variable cible: (3134,)


In [17]:
# Cellule 13 : Entraînement Random Forest
# Séparation train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Création et entraînement du modèle
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("📈 PERFORMANCES DU MODÈLE:")
print(f"MAE (Erreur Absolue Moyenne): {mae:,.0f} DH")
print(f"R² (Score de précision): {r2:.2%}")

📈 PERFORMANCES DU MODÈLE:
MAE (Erreur Absolue Moyenne): 633,594 DH
R² (Score de précision): 94.07%


In [18]:
# 🛠️ CELLULE : CRÉER LE DOSSIER MODELS
import os

# Créer le dossier models s'il n'existe pas
os.makedirs('../models', exist_ok=True)

# Vérifier
if os.path.exists('../models'):
    print("✅ Dossier 'models' créé avec succès!")
else:
    print("❌ Problème avec la création du dossier")

✅ Dossier 'models' créé avec succès!


In [19]:
# 🎯 CELLULE FINALE : CRÉER LE VRAI MODÈLE ML

# 1. Importer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib
import os

# 2. S'assurer que le dossier models existe
os.makedirs('../models', exist_ok=True)

# 3. Préparer les données
features = ['surface', 'pieces', 'chambres', 'salles_bain', 'etage']
X = df_clean[features].fillna(df_clean[features].median())
y = df_clean['prix_numerique']

print(f"📊 Données pour l'entraînement : {X.shape}")

# 4. Entraîner le modèle
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# 5. Sauvegarder le modèle
chemin_modele = '../models/mon_modele_immobilier.pkl'
joblib.dump(model, 'mon_modele_immobilier.pkl')

print("✅ MODÈLE SAUVEGARDÉ !")
print(f"📁 Emplacement : {chemin_modele}")
print("🎯 Tu peux maintenant l'utiliser dans ton app Streamlit")

# 6. Tester une prédiction
exemple_prediction = model.predict([[100, 3, 2, 1, 2]])[0]
print(f"🧪 Test prédiction (100m², 3 pièces) : {exemple_prediction:,.0f} DH")

📊 Données pour l'entraînement : (3134, 5)
✅ MODÈLE SAUVEGARDÉ !
📁 Emplacement : ../models/mon_modele_immobilier.pkl
🎯 Tu peux maintenant l'utiliser dans ton app Streamlit
🧪 Test prédiction (100m², 3 pièces) : 1,066,186 DH


In [20]:
from sklearn.metrics import mean_absolute_error, r2_score

# Sur ton test set
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:,.0f} DH")
print(f"R²: {r2:.2%}")

MAE: 633,594 DH
R²: 94.07%
